# 04. Torch Export AOT Autograd and IR Inspection

- **Category:** 12_compilation_and_performance
- **Lineage & Citations:** Ansel et al. (2024 *PyTorch 2: Faster Machine Learning Through Dynamic Python Bytecode Transformation and Graph Compilation*); Suhan et al. (2022 *AOTAutograd: Capturing Ahead-of-Time Autograd Graphs*); Hendrycks & Gimpel (2016 *Gaussian Error Linear Units (GELUs)*).
- **Purpose:** Formulate and verify hermetic graph capture via `torch.export`, Core ATen primitive decomposition (Softmax, LayerNorm), and AOTAutograd ahead-of-time reverse-mode Vector-Jacobian Product (VJP) graph extraction.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Torch FX Graph Capture and Custom Compiler Passes](03_torch_fx_graph_capture_and_custom_compiler_passes.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. Torch Library Custom Ops and Fake Kernels](05_torch_library_custom_ops_and_fake_kernels.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not export programs containing dynamic side-effects, in-place tensor mutations across external references, or data-dependent shape loops without specifying explicit `torch.export.Dim` dynamic shape bounds.

## Mathematical Foundations & Core ATen Decompositions

1. **Core ATen Softmax**:
   $$\mathbf{m} = \operatorname{amax}(\mathbf{x}, -1), \quad \mathbf{e} = \exp(\mathbf{x} - \mathbf{m}), \quad \mathbf{y} = \mathbf{e} / \operatorname{sum}(\mathbf{e}, -1)$$
2. **Core ATen LayerNorm**:
   $$\boldsymbol{\mu} = \operatorname{mean}(\mathbf{x}), \quad \hat{\mathbf{x}} = (\mathbf{x} - \boldsymbol{\mu}) \cdot \operatorname{rsqrt}(\operatorname{var}(\mathbf{x}) + \epsilon), \quad \mathbf{y} = \hat{\mathbf{x}} \odot \boldsymbol{\gamma} + \boldsymbol{\beta}$$
3. **AOT Reverse-Mode VJP Adjoint Graph**:
   $$\mathbf{z} = \mathbf{x} \mathbf{W} + \mathbf{b}, \quad \mathbf{g}_z = \mathbf{g}_a \odot \left[ \frac{1}{2} \left(1 + \operatorname{erf}\left(\frac{\mathbf{z}}{\sqrt{2}}\right)\right) + \frac{\mathbf{z}}{\sqrt{2\pi}} e^{-\frac{\mathbf{z}^2}{2}} \right]$$
   $$\nabla_{\mathbf{W}} = \mathbf{x}^T \mathbf{g}_z, \quad \nabla_{\mathbf{b}} = \sum \mathbf{g}_z, \quad \nabla_{\mathbf{x}} = \mathbf{g}_z \mathbf{W}^T$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/12_compilation_and_performance/04_torch_export_aot_autograd_and_ir_inspection.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def aten_core_softmax(x, dim=-1):
    m = torch.amax(x, dim=dim, keepdim=True)
    e = torch.exp(x - m)
    s = torch.sum(e, dim=dim, keepdim=True)
    return e / s

def aten_core_layernorm(x, gamma, beta, eps=1e-5):
    mean = torch.mean(x, dim=-1, keepdim=True)
    var = torch.mean((x - mean) ** 2, dim=-1, keepdim=True)
    rsqrt_var = torch.rsqrt(var + float(eps))
    return (x - mean) * rsqrt_var * gamma + beta

def aot_forward_backward_affine_gelu(x, w, b, grad_out):
    z = x @ w + b
    sqrt_2 = 1.4142135623730951
    sqrt_2_pi = 2.5066282746310002
    a = 0.5 * z * (1.0 + torch.erf(z / sqrt_2))
    d_gelu_dz = 0.5 * (1.0 + torch.erf(z / sqrt_2)) + (z / sqrt_2_pi) * torch.exp(-0.5 * z * z)
    gz = grad_out * d_gelu_dz
    return a, x.t() @ gz, torch.sum(gz, dim=0), gz @ w.t()

In [ ]:
# Verify Core ATen decompositions against golden references
sm = aten_core_softmax(weights["sm_x"], dim=-1)
assert torch.max(torch.abs(sm - weights["sm_ref"])) < 1e-5

ln = aten_core_layernorm(weights["ln_x"], weights["ln_gamma"], weights["ln_beta"], eps=1e-5)
assert torch.max(torch.abs(ln - weights["ln_ref"])) < 1e-5

# Verify AOTAutograd forward and backward VJP adjoints
fwd_a, grad_w, grad_b, grad_x = aot_forward_backward_affine_gelu(
    weights["mlp_x"], weights["mlp_w"], weights["mlp_b"], weights["mlp_grad_out"]
)
assert torch.max(torch.abs(fwd_a - weights["fwd_a_ref"])) < 1e-5
assert torch.max(torch.abs(grad_w - weights["grad_w_ref"])) < 1e-5
assert torch.max(torch.abs(grad_b - weights["grad_b_ref"])) < 1e-5
assert torch.max(torch.abs(grad_x - weights["grad_x_ref"])) < 1e-5
print("All Core ATen and AOTAutograd parity assertions verified!")

In [ ]:
# Microbenchmark: AOT Autograd VJP graph execution
x, w, b, g = weights["mlp_x"], weights["mlp_w"], weights["mlp_b"], weights["mlp_grad_out"]
t0 = time.perf_counter()
for _ in range(500):
    _ = aot_forward_backward_affine_gelu(x, w, b, g)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Joint AOT Forward + Reverse VJP latency: {lat_us:.2f} us")

In [ ]:
# Equivalence check: Compare AOT manual VJP against PyTorch autograd engine
x_dyn = weights["mlp_x"].clone().requires_grad_(True)
w_dyn = weights["mlp_w"].clone().requires_grad_(True)
b_dyn = weights["mlp_b"].clone().requires_grad_(True)
z_dyn = x_dyn @ w_dyn + b_dyn
a_dyn = 0.5 * z_dyn * (1.0 + torch.erf(z_dyn / (2.0 ** 0.5)))
a_dyn.backward(weights["mlp_grad_out"])

err_w = float(torch.max(torch.abs(grad_w - w_dyn.grad)))
err_b = float(torch.max(torch.abs(grad_b - b_dyn.grad)))
err_x = float(torch.max(torch.abs(grad_x - x_dyn.grad)))
print(f"Autograd match errors: W={err_w:.2e}, b={err_b:.2e}, x={err_x:.2e}")
assert max(err_w, err_b, err_x) < 1e-5, "AOT VJP failed autograd equivalence!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **`torch.export`** captures a hermetic computational graph with guaranteed static semantics, producing an `ExportedProgram` deployable via AOTInductor C++ runtimes.
2. **Core ATen IR** standardizes 2,000+ PyTorch APIs into ~180 side-effect-free primitives, simplifying downstream compiler optimizations.
3. **AOTAutograd** captures joint forward and reverse-mode VJP adjoint graphs ahead-of-time, enabling whole-program compiler optimizations across the backward pass.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_torch_export_aot_autograd_and_ir_inspection.md](../../markdown/12_compilation_and_performance/04_torch_export_aot_autograd_and_ir_inspection.md)
- **PyTorch 2.x Module:** [04_torch_export_aot_autograd_and_ir_inspection.py](../../pytorch2/12_compilation_and_performance/04_torch_export_aot_autograd_and_ir_inspection.py)
- **Functional JAX Module:** [04_torch_export_aot_autograd_and_ir_inspection.py](../../jax/12_compilation_and_performance/04_torch_export_aot_autograd_and_ir_inspection.py)
- **Native Rust Candle Track:** [04_torch_export_aot_autograd_and_ir_inspection.rs](../../candle/12_compilation_and_performance/04_torch_export_aot_autograd_and_ir_inspection.rs)
- **Hardware Kernel (CPU / WGSL):** [04_torch_export_aot_autograd_and_ir_inspection.rs](../../kernels/12_compilation_and_performance/04_torch_export_aot_autograd_and_ir_inspection.rs)
- **Golden SafeTensors:** [04_torch_export_aot_autograd_and_ir_inspection.safetensors](../../weights/12_compilation_and_performance/04_torch_export_aot_autograd_and_ir_inspection.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Torch FX Graph Capture and Custom Compiler Passes](03_torch_fx_graph_capture_and_custom_compiler_passes.ipynb) | [Master Index](../../README.md#12_compilation_and_performance) | [Next: 05. Torch Library Custom Ops and Fake Kernels](05_torch_library_custom_ops_and_fake_kernels.ipynb) &rarr;